# Standalone Experiment C — Spike Encoder Benchmark

This notebook benchmarks **multiple Action 0 + Action 1 encoder combinations** using an independent, reconstruction-only Experiment C protocol:

- **No Experiment A checkpoint is required.**
- The **user split is fixed once** and reused by every run.
- Five **training/evaluation random seeds** are used as the variation source.
- Three matched CNN probes are evaluated: `cnn_s`, `cnn_m`, `cnn_l`.
- Every encoder combination must pass a **cross-combination preflight** before training.
- The standard Experiment C evaluation is reused; this notebook only adds orchestration and cross-combination aggregation.

Primary comparison unit:

\[
\text{encoder combination} \times \text{CNN probe} \times \text{training seed}
\]

Recommended interpretation:

- **Encoder combination** = treatment being compared.
- **CNN probe** = model-capacity factor; report separately rather than averaging probes together.
- **Training seed** = replicate / variation source; error bars are mean ± sample SD across seeds.

The notebook also produces:

1. selected-metric mean ± SD comparison plots;
2. mean row-normalized confusion-matrix **small multiples** for every probe;
3. per-class recall heatmaps across encoder combinations;
4. optional confusion-matrix SD small multiples to visualize seed instability.

## 1. Imports and repository root

Run this notebook from the repository checkout. The helper below finds the `writingRing` root when the notebook is opened from `notebooks/` or another subdirectory.

In [ ]:
from __future__ import annotations

from dataclasses import replace
from pathlib import Path
import json
import math
import re
import sys
from typing import Mapping, Sequence

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display


def find_repository_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    candidates = [start, *start.parents]
    for candidate in candidates:
        if (candidate / "pyproject.toml").is_file() and (candidate / "snn").is_dir():
            return candidate
    raise FileNotFoundError(
        "Could not locate the writingRing repository root. "
        "Open this notebook from inside the repository checkout."
    )


REPO_ROOT = find_repository_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from snn.accel_reconstruction_eval import (
    experiment_c_config,
    load_acceleration_data,
    prepare_user_disjoint_splits,
    restrict_manifest_to_cohort,
)
from snn.accel_reconstruction_eval.config import UserSplitConfig
from scripts.run_experiment_c import run_experiment_c

print("Repository root:", REPO_ROOT)

## 2. Benchmark configuration

Edit this cell before running the benchmark.

### Encoder combinations

Each dictionary entry defines **one treatment**. Its value must contain exactly two roots in this order:

1. Action 0 combination root
2. Action 1 combination root

The roots can point at the combination root or at the resolved `segmentation_padded/` directory; the repository loader resolves the padded dataset.

The **first combination** is the reference for the canonical cohort and fixed user split. This does **not** make it a performance baseline unless you choose to interpret it that way.

### Seeds

`SPLIT_SEED` is used only once to choose the fixed user split. `TRAINING_SEEDS` are the five repeated-training seeds and are the only seed variation summarized in the final error bars.

In [ ]:
# -----------------------------------------------------------------------------
# USER CONFIGURATION
# -----------------------------------------------------------------------------

COMBINATIONS: dict[str, tuple[Path, Path]] = {
    # Reference / canonical cohort source. Add more combinations below.
    "baseline": (
        REPO_ROOT / "outputs/action0_rectified/low-pass/aligned-board-events",
        REPO_ROOT / "outputs/action1_rectified/low-pass/aligned-board-events",
    ),

    # Example shape only — replace with your real encoder variants:
    # "wavelets_1_2_3_4_5": (
    #     REPO_ROOT / "outputs/reencoded_wavelet_variants/action0_...",
    #     REPO_ROOT / "outputs/reencoded_wavelet_variants/action1_...",
    # ),
}

PROBES: tuple[str, ...] = ("cnn_s", "cnn_m", "cnn_l")
TRAINING_SEEDS: tuple[int, ...] = (13, 37, 71, 101, 137)
SPLIT_SEED: int = 12345

# Cohort controls applied before the fixed split is created.
# Keep user_17 excluded if you want to follow the repository's current caution.
EXCLUDED_USERS: tuple[str, ...] = ("user_17",)
INCLUDED_LABELS: tuple[str, ...] | None = None
TRAIN_FRACTION: float = 0.70
VAL_FRACTION: float = 0.15
REQUIRE_ALL_LABELS_IN_ALL_SPLITS: bool = True

# Selected metrics to aggregate across the five training seeds.
# Keys are display labels; values are columns in Experiment C summary.csv.
SELECTED_METRICS: dict[str, str] = {
    "CNN balanced accuracy": "CNN_test_balanced_accuracy",
    "CNN macro-F1": "CNN_test_macro_f1",
    "Retrieval macro mAP": "retrieval_macro_mAP",
    "SameLabel@1 macro": "SameLabel_macro_at_1",
    "D_inter / D_intra": "D_inter_over_D_intra",
    "Silhouette macro": "silhouette_macro",
}

OUTPUT_ROOT = REPO_ROOT / "notebooks/artifacts/experiment_C_encoder_benchmark"
FIGURE_ROOT = OUTPUT_ROOT / "comparison_figures"

# None = let run_experiment_c choose CUDA when configured/available.
DEVICE: str | None = None

# If True, existing run directories with summary.csv + embeddings_test.npz are reused.
RESUME: bool = True

# Expensive but strongest preflight: require raw IMU channels 15:21 to be exactly
# identical across encoder combinations, package by package. Event channels 0:15
# are intentionally allowed to differ.
STRICT_RAW_IMU_EQUAL: bool = True
RAW_IMU_COMPARE_SEGMENT_CHUNK: int = 64

# Set True only when you are ready to launch all trainings.
RUN_BENCHMARK: bool = False

# Visualization controls.
ANNOTATE_CONFUSION_IF_CLASSES_LEQ: int = 12
PLOT_CONFUSION_STD: bool = True

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
FIGURE_ROOT.mkdir(parents=True, exist_ok=True)

print("Combinations:", list(COMBINATIONS))
print("Training seeds:", TRAINING_SEEDS)
print("Probes:", PROBES)
print("Output root:", OUTPUT_ROOT)

## 3. Cross-combination preflight and fixed split

This preflight enforces the benchmark's causal comparison assumptions before any training begins.

For every combination it checks:

- exactly two roots are supplied;
- reconstruction artifacts load and validate;
- the selected Action set matches the reference;
- target length and sampling rate match the reference;
- the **canonical sample IDs are identical** after applying the fixed cohort;
- user/action/label and, when available, `valid_length` agree sample-by-sample;
- package keys, labels, valid lengths, and valid masks agree;
- optionally, SpikeIMU channels `15:21` are exactly equal across combinations.

The last check is useful when the intended treatment difference is the spike encoder only: it verifies that the copied raw acceleration/gyro channels did not change while event channels `0:15` are allowed to differ.

If any check fails, the benchmark stops instead of silently intersecting datasets.

In [ ]:
def safe_slug(text: str) -> str:
    slug = re.sub(r"[^A-Za-z0-9._-]+", "_", str(text)).strip("._-")
    if not slug:
        raise ValueError(f"Combination name cannot be converted to a safe path: {text!r}")
    return slug


def _semantic_manifest(frame: pd.DataFrame) -> pd.DataFrame:
    required = ["sample_id", "user", "action", "label"]
    missing = [column for column in required if column not in frame.columns]
    if missing:
        raise KeyError(f"sample_manifest is missing required columns: {missing}")

    columns = required.copy()
    if "valid_length" in frame.columns:
        columns.append("valid_length")

    result = frame.loc[:, columns].copy()
    result["sample_id"] = result["sample_id"].astype(str)
    result["user"] = result["user"].astype(str)
    result["action"] = result["action"].astype(str)
    result["label"] = result["label"].astype(str)
    result = result.sort_values("sample_id", kind="stable").reset_index(drop=True)
    if result["sample_id"].duplicated().any():
        duplicates = result.loc[result["sample_id"].duplicated(), "sample_id"].tolist()
        raise ValueError(f"Duplicate sample IDs found: {duplicates[:10]}")
    return result


def _package_map(data) -> dict[tuple[str, str], object]:
    mapping: dict[tuple[str, str], object] = {}
    for package in data.packages:
        key = (str(package.user), str(package.action))
        if key in mapping:
            raise ValueError(f"Duplicate package key: {key}")
        mapping[key] = package
    return mapping


def _assert_raw_imu_equal(reference_package, current_package, chunk_segments: int) -> None:
    ref = reference_package.padded_spike_imu
    cur = current_package.padded_spike_imu
    if ref.shape != cur.shape:
        raise AssertionError(
            f"Raw SpikeIMU shape mismatch for {(reference_package.user, reference_package.action)}: "
            f"{ref.shape} vs {cur.shape}"
        )
    for start in range(0, ref.shape[0], chunk_segments):
        stop = min(start + chunk_segments, ref.shape[0])
        if not np.array_equal(ref[start:stop, :, 15:21], cur[start:stop, :, 15:21]):
            raise AssertionError(
                "Raw IMU channels 15:21 differ for package "
                f"{(reference_package.user, reference_package.action)} "
                f"at segment slice [{start}:{stop}]"
            )


def run_preflight(
    combinations: Mapping[str, Sequence[Path]],
) -> tuple[dict[str, object], object, UserSplitConfig, pd.DataFrame]:
    if not combinations:
        raise ValueError("COMBINATIONS must contain at least one entry")
    if len(set(combinations)) != len(combinations):
        raise ValueError("Combination names must be unique")

    loaded: dict[str, object] = {}
    for name, roots in combinations.items():
        if len(tuple(roots)) != 2:
            raise ValueError(f"{name!r} must contain exactly two roots: Action 0 then Action 1")
        loaded[name] = load_acceleration_data(
            list(roots),
            repository_root=REPO_ROOT,
            require_reconstruction=True,
        )

    reference_name = next(iter(combinations))
    reference_data = loaded[reference_name]

    reference_split = prepare_user_disjoint_splits(
        reference_data.sample_manifest,
        train_fraction=TRAIN_FRACTION,
        val_fraction=VAL_FRACTION,
        seed=SPLIT_SEED,
        excluded_users=EXCLUDED_USERS,
        included_labels=INCLUDED_LABELS,
        require_all_users_assigned=True,
        require_all_labels_in_all_splits=REQUIRE_ALL_LABELS_IN_ALL_SPLITS,
    )

    fixed_split_config = UserSplitConfig(
        explicit_train_users=tuple(reference_split.train_users),
        explicit_val_users=tuple(reference_split.val_users),
        explicit_test_users=tuple(reference_split.test_users),
        require_all_users_assigned=True,
        require_all_labels_in_all_splits=REQUIRE_ALL_LABELS_IN_ALL_SPLITS,
        excluded_users=EXCLUDED_USERS,
        included_labels=INCLUDED_LABELS,
    )
    fixed_split_config.validate()

    split_users = (
        *reference_split.train_users,
        *reference_split.val_users,
        *reference_split.test_users,
    )
    canonical = restrict_manifest_to_cohort(
        reference_data.sample_manifest,
        split_users=split_users,
        class_to_idx=reference_split.class_to_idx,
    )
    canonical_semantic = _semantic_manifest(canonical)
    canonical_ids = tuple(canonical_semantic["sample_id"].tolist())

    reference_actions = tuple(reference_data.selected_actions)
    reference_target_length = int(reference_data.producer_metadata.target_length)
    reference_sampling_rate = float(reference_data.producer_metadata.sampling_rate_hz)
    reference_packages = _package_map(reference_data)

    rows: list[dict[str, object]] = []

    for name, data in loaded.items():
        current = restrict_manifest_to_cohort(
            data.sample_manifest,
            split_users=split_users,
            class_to_idx=reference_split.class_to_idx,
        )
        current_semantic = _semantic_manifest(current)
        current_ids = tuple(current_semantic["sample_id"].tolist())

        if current_ids != canonical_ids:
            reference_set = set(canonical_ids)
            current_set = set(current_ids)
            missing = sorted(reference_set - current_set)[:20]
            extra = sorted(current_set - reference_set)[:20]
            raise AssertionError(
                f"Cross-combination cohort mismatch for {name!r}. "
                f"Missing IDs (first 20): {missing}; extra IDs (first 20): {extra}"
            )

        compare_columns = [
            column
            for column in ("sample_id", "user", "action", "label", "valid_length")
            if column in canonical_semantic.columns and column in current_semantic.columns
        ]
        if not canonical_semantic[compare_columns].equals(current_semantic[compare_columns]):
            raise AssertionError(
                f"Semantic manifest mismatch for {name!r} in columns {compare_columns}"
            )

        actions = tuple(data.selected_actions)
        if actions != reference_actions:
            raise AssertionError(
                f"Action-set mismatch for {name!r}: {actions} vs {reference_actions}"
            )
        if int(data.producer_metadata.target_length) != reference_target_length:
            raise AssertionError(
                f"target_length mismatch for {name!r}: "
                f"{data.producer_metadata.target_length} vs {reference_target_length}"
            )
        if not np.isclose(float(data.producer_metadata.sampling_rate_hz), reference_sampling_rate):
            raise AssertionError(
                f"sampling_rate_hz mismatch for {name!r}: "
                f"{data.producer_metadata.sampling_rate_hz} vs {reference_sampling_rate}"
            )

        current_packages = _package_map(data)
        if current_packages.keys() != reference_packages.keys():
            raise AssertionError(f"Package-key mismatch for {name!r}")

        if name != reference_name:
            for key in reference_packages:
                ref_package = reference_packages[key]
                cur_package = current_packages[key]
                if not np.array_equal(ref_package.labels, cur_package.labels):
                    raise AssertionError(f"Label-array mismatch for {name!r}, package {key}")
                if not np.array_equal(ref_package.valid_lengths, cur_package.valid_lengths):
                    raise AssertionError(f"valid_lengths mismatch for {name!r}, package {key}")
                if not np.array_equal(ref_package.valid_mask, cur_package.valid_mask):
                    raise AssertionError(f"valid_mask mismatch for {name!r}, package {key}")
                if STRICT_RAW_IMU_EQUAL:
                    _assert_raw_imu_equal(
                        ref_package,
                        cur_package,
                        chunk_segments=RAW_IMU_COMPARE_SEGMENT_CHUNK,
                    )

        encoder_hashes = sorted(
            {
                str(meta.spike_encoder_spec_sha256)
                for meta in data.producer_metadatas
                if meta.spike_encoder_spec_sha256 is not None
            }
        )
        rows.append(
            {
                "combination": name,
                "root_count": len(data.padded_roots),
                "actions": ",".join(actions),
                "selected_samples": len(current_semantic),
                "selected_users": len(set(current_semantic["user"])),
                "selected_labels": len(set(current_semantic["label"])),
                "target_length": int(data.producer_metadata.target_length),
                "sampling_rate_hz": float(data.producer_metadata.sampling_rate_hz),
                "encoder_spec_hashes": " | ".join(encoder_hashes),
                "strict_raw_imu_equal_checked": bool(STRICT_RAW_IMU_EQUAL),
            }
        )

    preflight_table = pd.DataFrame(rows)
    return loaded, reference_split, fixed_split_config, preflight_table


LOADED_COMBINATIONS, FIXED_SPLIT, FIXED_SPLIT_CONFIG, PREFLIGHT_TABLE = run_preflight(COMBINATIONS)

display(PREFLIGHT_TABLE)
print("Fixed train users:", FIXED_SPLIT.train_users)
print("Fixed val users:", FIXED_SPLIT.val_users)
print("Fixed test users:", FIXED_SPLIT.test_users)
print("Class mapping:", FIXED_SPLIT.class_to_idx)

## 4. Persist the benchmark design

The design file records the fixed split and the exact treatment definitions before training starts. This makes the later aggregate tables auditable.

In [ ]:
DESIGN_PATH = OUTPUT_ROOT / "benchmark_design.json"
PREFLIGHT_CSV = OUTPUT_ROOT / "cross_combination_preflight.csv"

benchmark_design = {
    "protocol": "standalone_experiment_c_encoder_benchmark_v1",
    "reference_combination": next(iter(COMBINATIONS)),
    "combinations": {
        name: [str(Path(root).resolve()) for root in roots]
        for name, roots in COMBINATIONS.items()
    },
    "split_seed": SPLIT_SEED,
    "training_seeds": list(TRAINING_SEEDS),
    "probes": list(PROBES),
    "excluded_users": list(EXCLUDED_USERS),
    "included_labels": None if INCLUDED_LABELS is None else list(INCLUDED_LABELS),
    "train_users": list(FIXED_SPLIT.train_users),
    "val_users": list(FIXED_SPLIT.val_users),
    "test_users": list(FIXED_SPLIT.test_users),
    "class_to_idx": FIXED_SPLIT.class_to_idx,
    "selected_metrics": SELECTED_METRICS,
    "strict_raw_imu_equal": STRICT_RAW_IMU_EQUAL,
}

DESIGN_PATH.write_text(json.dumps(benchmark_design, indent=2), encoding="utf-8")
PREFLIGHT_TABLE.to_csv(PREFLIGHT_CSV, index=False)
print("Saved:", DESIGN_PATH)
print("Saved:", PREFLIGHT_CSV)

## 5. Run all standalone Experiment C trainings

Every run uses:

- `reference_checkpoint=None`
- `allow_new_split=True`
- the same explicit train/validation/test user lists
- reconstruction-only train/validation/test
- reconstruction-train normalization
- one of the three probe variants
- one of the five training/evaluation seeds

Output layout:

```text
notebooks/artifacts/experiment_C_encoder_benchmark/
  <combination>/
    seed_<seed>/
      cnn_s/
      cnn_m/
      cnn_l/
```

`run_experiment_c()` appends the probe directory itself, so each standard C artifact layout remains intact.

In [ ]:
MASTER_RESULTS_PATH = OUTPUT_ROOT / "master_results.csv"


def build_c_config(seed: int, probe: str, output_base: Path):
    config = experiment_c_config(
        output_dir=output_base,
        random_seed=int(seed),
        probe_variant=probe,
    )
    config = replace(config, split=FIXED_SPLIT_CONFIG)
    config.validate()
    return config


def expected_run_dir(combination: str, seed: int, probe: str) -> Path:
    return OUTPUT_ROOT / safe_slug(combination) / f"seed_{seed}" / probe


def load_existing_summary(run_dir: Path) -> pd.DataFrame:
    path = run_dir / "summary.csv"
    if not path.is_file():
        raise FileNotFoundError(path)
    frame = pd.read_csv(path)
    if len(frame) != 1:
        raise ValueError(f"Expected one summary row in {path}, found {len(frame)}")
    return frame


def run_benchmark() -> pd.DataFrame:
    records: list[dict[str, object]] = []
    total = len(COMBINATIONS) * len(PROBES) * len(TRAINING_SEEDS)
    ordinal = 0

    for combination, roots in COMBINATIONS.items():
        combination_base = OUTPUT_ROOT / safe_slug(combination)
        for probe in PROBES:
            for seed in TRAINING_SEEDS:
                ordinal += 1
                run_base = combination_base / f"seed_{seed}"
                run_dir = run_base / probe
                summary_path = run_dir / "summary.csv"
                embedding_path = run_dir / "embeddings_test.npz"

                print(f"[{ordinal}/{total}] {combination} | {probe} | seed={seed}")

                if RESUME and summary_path.is_file() and embedding_path.is_file():
                    summary = load_existing_summary(run_dir)
                else:
                    config = build_c_config(seed, probe, run_base)
                    result = run_experiment_c(
                        root=list(roots),
                        repository_root=REPO_ROOT,
                        output_dir=run_base,
                        reference_checkpoint=None,
                        config=config,
                        device=DEVICE,
                        allow_new_split=True,
                        probe_variant=probe,
                    )
                    summary = result.summary.copy()

                row = summary.iloc[0].to_dict()
                row.update(
                    {
                        "combination": combination,
                        "combination_slug": safe_slug(combination),
                        "probe": probe,
                        "training_seed": int(seed),
                        "run_dir": str(run_dir),
                    }
                )
                records.append(row)

                # Incremental checkpoint of benchmark progress.
                pd.DataFrame(records).to_csv(MASTER_RESULTS_PATH, index=False)

    result_frame = pd.DataFrame(records)
    result_frame.to_csv(MASTER_RESULTS_PATH, index=False)
    return result_frame


if RUN_BENCHMARK:
    MASTER_RESULTS = run_benchmark()
else:
    if MASTER_RESULTS_PATH.is_file():
        MASTER_RESULTS = pd.read_csv(MASTER_RESULTS_PATH)
        print("RUN_BENCHMARK=False; loaded existing master results:", MASTER_RESULTS_PATH)
    else:
        MASTER_RESULTS = pd.DataFrame()
        print(
            "RUN_BENCHMARK=False and no master_results.csv exists yet. "
            "Set RUN_BENCHMARK=True in the configuration cell when ready."
        )

display(MASTER_RESULTS.head() if not MASTER_RESULTS.empty else MASTER_RESULTS)

## 6. Aggregate selected metrics across training seeds

For every **combination × probe × metric**, report:

- mean across the five training seeds;
- sample standard deviation (`ddof=1`);
- number of completed seeds.

The plots use the seed SD as the variation/error bar. Probes are kept separate instead of being averaged together.

In [ ]:
def require_complete_master_results(frame: pd.DataFrame) -> None:
    if frame.empty:
        raise RuntimeError("No benchmark results are available. Run the training cell first.")

    required = {"combination", "probe", "training_seed", *SELECTED_METRICS.values()}
    missing = sorted(required.difference(frame.columns))
    if missing:
        raise KeyError(f"master_results.csv is missing required columns: {missing}")

    expected = {
        (combination, probe, int(seed))
        for combination in COMBINATIONS
        for probe in PROBES
        for seed in TRAINING_SEEDS
    }
    observed = {
        (str(row.combination), str(row.probe), int(row.training_seed))
        for row in frame.loc[:, ["combination", "probe", "training_seed"]].itertuples(index=False)
    }
    missing_runs = sorted(expected - observed)
    if missing_runs:
        raise RuntimeError(f"Benchmark is incomplete; missing {len(missing_runs)} runs: {missing_runs[:20]}")


if not MASTER_RESULTS.empty:
    require_complete_master_results(MASTER_RESULTS)

    long_frames = []
    for display_name, column in SELECTED_METRICS.items():
        part = MASTER_RESULTS.loc[:, ["combination", "probe", "training_seed", column]].copy()
        part = part.rename(columns={column: "value"})
        part["metric"] = display_name
        part["metric_column"] = column
        long_frames.append(part)
    METRIC_LONG = pd.concat(long_frames, ignore_index=True)

    METRIC_AGGREGATE = (
        METRIC_LONG
        .groupby(["metric", "metric_column", "combination", "probe"], as_index=False)
        .agg(mean=("value", "mean"), sd=("value", "std"), n=("value", "count"))
    )
    METRIC_AGGREGATE.to_csv(OUTPUT_ROOT / "selected_metric_aggregate.csv", index=False)
    display(METRIC_AGGREGATE)
else:
    METRIC_LONG = pd.DataFrame()
    METRIC_AGGREGATE = pd.DataFrame()

In [ ]:
def plot_selected_metric_comparisons(
    aggregate: pd.DataFrame,
    *,
    output_dir: Path = FIGURE_ROOT,
) -> dict[str, Path]:
    if aggregate.empty:
        raise RuntimeError("No aggregate metrics to plot")

    output_dir.mkdir(parents=True, exist_ok=True)
    saved: dict[str, Path] = {}
    combination_order = list(COMBINATIONS)
    x = np.arange(len(combination_order), dtype=float)
    width = 0.8 / max(1, len(PROBES))

    for metric_name in SELECTED_METRICS:
        subset = aggregate.loc[aggregate["metric"] == metric_name].copy()
        fig, ax = plt.subplots(figsize=(max(9, 1.4 * len(combination_order)), 5.4))

        for probe_index, probe in enumerate(PROBES):
            probe_table = (
                subset.loc[subset["probe"] == probe]
                .set_index("combination")
                .reindex(combination_order)
            )
            if probe_table["mean"].isna().any():
                raise ValueError(f"Missing aggregate rows for {metric_name!r}, probe={probe!r}")

            offsets = x - 0.4 + width / 2 + probe_index * width
            ax.bar(
                offsets,
                probe_table["mean"].to_numpy(dtype=float),
                width=width,
                yerr=probe_table["sd"].to_numpy(dtype=float),
                capsize=4,
                label=probe,
            )

        ax.set_xticks(x)
        ax.set_xticklabels(combination_order, rotation=25, ha="right")
        ax.set_ylabel(metric_name)
        ax.set_title(f"{metric_name}: mean ± SD across {len(TRAINING_SEEDS)} training seeds")
        ax.grid(axis="y", alpha=0.25)
        ax.legend(title="CNN probe")
        fig.tight_layout()

        filename = f"metric_{safe_slug(metric_name)}.png"
        path = output_dir / filename
        fig.savefig(path, dpi=180, bbox_inches="tight")
        plt.show()
        saved[metric_name] = path

    return saved


if not METRIC_AGGREGATE.empty:
    METRIC_FIGURES = plot_selected_metric_comparisons(METRIC_AGGREGATE)
    display(pd.DataFrame({"metric": METRIC_FIGURES.keys(), "figure": METRIC_FIGURES.values()}))

## 7. Aggregate confusion matrices across seeds

For each completed run, the saved `embeddings_test.npz` contains the true labels `y` and CNN predictions `cnn_pred`.

For each seed we compute a **row-normalized confusion matrix**:

\[
C_{ij} = P(\hat y=j \mid y=i)
\]

Then, for each **combination × probe**, the notebook computes the element-wise mean and SD across training seeds.

Why row normalization:

- the diagonal is per-class recall;
- off-diagonal entries are conditional confusion rates;
- comparisons remain interpretable even if class sample counts differ.

The fixed split and cross-combination cohort check ensure these matrices are paired on the same held-out users and canonical sample cohort.

In [ ]:
IDX_TO_CLASS = {index: label for label, index in FIXED_SPLIT.class_to_idx.items()}
CLASS_LABELS = [IDX_TO_CLASS[index] for index in range(len(IDX_TO_CLASS))]
NUM_CLASSES = len(CLASS_LABELS)


def row_normalized_confusion(y_true: np.ndarray, y_pred: np.ndarray, num_classes: int) -> np.ndarray:
    y_true = np.asarray(y_true, dtype=np.int64)
    y_pred = np.asarray(y_pred, dtype=np.int64)
    if y_true.shape != y_pred.shape:
        raise ValueError(f"Shape mismatch: y_true={y_true.shape}, y_pred={y_pred.shape}")

    counts = np.zeros((num_classes, num_classes), dtype=np.int64)
    np.add.at(counts, (y_true, y_pred), 1)
    row_totals = counts.sum(axis=1, keepdims=True)
    if np.any(row_totals == 0):
        empty_classes = np.flatnonzero(row_totals[:, 0] == 0).tolist()
        raise ValueError(f"Test split has zero samples for class indices: {empty_classes}")
    return counts / row_totals


def load_test_predictions(run_dir: Path) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    path = run_dir / "embeddings_test.npz"
    if not path.is_file():
        raise FileNotFoundError(path)
    with np.load(path, allow_pickle=False) as payload:
        required = {"y", "cnn_pred", "sample_id"}
        missing = sorted(required.difference(payload.files))
        if missing:
            raise KeyError(f"{path} is missing arrays: {missing}")
        return (
            payload["y"].astype(np.int64, copy=False),
            payload["cnn_pred"].astype(np.int64, copy=False),
            payload["sample_id"].astype(str),
        )


def aggregate_confusions() -> tuple[dict[tuple[str, str], dict[str, np.ndarray]], pd.DataFrame]:
    aggregate: dict[tuple[str, str], dict[str, np.ndarray]] = {}
    recall_rows: list[dict[str, object]] = []

    for combination in COMBINATIONS:
        for probe in PROBES:
            matrices: list[np.ndarray] = []
            reference_ids: np.ndarray | None = None
            reference_y: np.ndarray | None = None

            for seed in TRAINING_SEEDS:
                run_dir = expected_run_dir(combination, seed, probe)
                y_true, y_pred, sample_ids = load_test_predictions(run_dir)

                # Seeds must evaluate the same fixed test cohort in the same order.
                if reference_ids is None:
                    reference_ids = sample_ids
                    reference_y = y_true
                else:
                    if not np.array_equal(reference_ids, sample_ids):
                        raise AssertionError(
                            f"Test sample ordering changed across seeds for {combination}, {probe}"
                        )
                    if not np.array_equal(reference_y, y_true):
                        raise AssertionError(
                            f"Test labels changed across seeds for {combination}, {probe}"
                        )

                matrix = row_normalized_confusion(y_true, y_pred, NUM_CLASSES)
                matrices.append(matrix)

                for class_index, class_label in enumerate(CLASS_LABELS):
                    recall_rows.append(
                        {
                            "combination": combination,
                            "probe": probe,
                            "training_seed": int(seed),
                            "class_index": class_index,
                            "class_label": class_label,
                            "recall": float(matrix[class_index, class_index]),
                        }
                    )

            stack = np.stack(matrices, axis=0)
            aggregate[(combination, probe)] = {
                "mean": stack.mean(axis=0),
                "sd": stack.std(axis=0, ddof=1),
                "all": stack,
            }

    recall_table = pd.DataFrame(recall_rows)
    recall_table.to_csv(OUTPUT_ROOT / "per_class_recall_by_seed.csv", index=False)
    return aggregate, recall_table


if not MASTER_RESULTS.empty:
    CONFUSION_AGGREGATE, PER_CLASS_RECALL = aggregate_confusions()
    display(PER_CLASS_RECALL.head())
else:
    CONFUSION_AGGREGATE = {}
    PER_CLASS_RECALL = pd.DataFrame()

## 8. Confusion-matrix small multiples

One figure is produced per CNN probe. Every subplot uses:

- the same class order;
- row normalization;
- the same 0–1 color scale;
- the mean confusion matrix across the five training seeds.

If the number of classes is small enough, cell values are annotated. When `PLOT_CONFUSION_STD=True`, a second small-multiples figure shows the per-cell SD across seeds.

In [ ]:
def _subplot_grid(n: int) -> tuple[int, int]:
    cols = min(3, max(1, n))
    rows = math.ceil(n / cols)
    return rows, cols


def plot_confusion_small_multiples(
    aggregate: Mapping[tuple[str, str], Mapping[str, np.ndarray]],
    *,
    statistic: str = "mean",
    output_dir: Path = FIGURE_ROOT,
) -> dict[str, Path]:
    if statistic not in {"mean", "sd"}:
        raise ValueError("statistic must be 'mean' or 'sd'")

    saved: dict[str, Path] = {}
    combinations = list(COMBINATIONS)
    annotate = NUM_CLASSES <= ANNOTATE_CONFUSION_IF_CLASSES_LEQ

    if statistic == "mean":
        vmin, vmax = 0.0, 1.0
    else:
        max_sd = max(
            float(np.nanmax(aggregate[(combination, probe)]["sd"]))
            for combination in combinations
            for probe in PROBES
        )
        vmin, vmax = 0.0, max(max_sd, 1e-12)

    for probe in PROBES:
        rows, cols = _subplot_grid(len(combinations))
        fig, axes = plt.subplots(
            rows,
            cols,
            figsize=(5.1 * cols, 4.7 * rows),
            squeeze=False,
            constrained_layout=True,
        )
        image = None

        for index, combination in enumerate(combinations):
            ax = axes[index // cols][index % cols]
            matrix = np.asarray(aggregate[(combination, probe)][statistic], dtype=float)
            image = ax.imshow(matrix, vmin=vmin, vmax=vmax, aspect="auto")
            ax.set_title(combination)
            ax.set_xlabel("Predicted class")
            ax.set_ylabel("True class")
            ax.set_xticks(np.arange(NUM_CLASSES))
            ax.set_yticks(np.arange(NUM_CLASSES))
            ax.set_xticklabels(CLASS_LABELS, rotation=90)
            ax.set_yticklabels(CLASS_LABELS)

            if annotate:
                for i in range(NUM_CLASSES):
                    for j in range(NUM_CLASSES):
                        ax.text(j, i, f"{matrix[i, j]:.2f}", ha="center", va="center", fontsize=7)

        for index in range(len(combinations), rows * cols):
            axes[index // cols][index % cols].axis("off")

        title = (
            f"{probe}: mean row-normalized confusion across {len(TRAINING_SEEDS)} seeds"
            if statistic == "mean"
            else f"{probe}: row-normalized confusion SD across {len(TRAINING_SEEDS)} seeds"
        )
        fig.suptitle(title)
        if image is not None:
            fig.colorbar(image, ax=axes.ravel().tolist(), shrink=0.78, label=statistic)

        path = output_dir / f"confusion_small_multiples_{probe}_{statistic}.png"
        fig.savefig(path, dpi=180, bbox_inches="tight")
        plt.show()
        saved[probe] = path

    return saved


if CONFUSION_AGGREGATE:
    CONFUSION_MEAN_FIGURES = plot_confusion_small_multiples(CONFUSION_AGGREGATE, statistic="mean")
    if PLOT_CONFUSION_STD:
        CONFUSION_SD_FIGURES = plot_confusion_small_multiples(CONFUSION_AGGREGATE, statistic="sd")

## 9. Per-class recall heatmaps

The diagonal of each row-normalized confusion matrix is class recall.

For every CNN probe, this notebook builds a heatmap with:

- **rows** = classes;
- **columns** = encoder combinations;
- **cell value** = mean recall across the five training seeds.

A second optional heatmap shows recall SD across seeds. This view scales much better than full confusion matrices when many encoder combinations are compared.

In [ ]:
def aggregate_per_class_recall(table: pd.DataFrame) -> pd.DataFrame:
    if table.empty:
        raise RuntimeError("No per-class recall records available")
    return (
        table
        .groupby(["probe", "combination", "class_index", "class_label"], as_index=False)
        .agg(
            mean_recall=("recall", "mean"),
            sd_recall=("recall", "std"),
            n=("recall", "count"),
        )
    )


def plot_per_class_recall_heatmaps(
    aggregate: pd.DataFrame,
    *,
    statistic: str = "mean_recall",
    output_dir: Path = FIGURE_ROOT,
) -> dict[str, Path]:
    if statistic not in {"mean_recall", "sd_recall"}:
        raise ValueError("statistic must be 'mean_recall' or 'sd_recall'")

    saved: dict[str, Path] = {}
    combination_order = list(COMBINATIONS)

    if statistic == "mean_recall":
        vmin, vmax = 0.0, 1.0
    else:
        max_sd = float(aggregate["sd_recall"].max())
        vmin, vmax = 0.0, max(max_sd, 1e-12)

    for probe in PROBES:
        subset = aggregate.loc[aggregate["probe"] == probe].copy()
        pivot = (
            subset.pivot(index="class_label", columns="combination", values=statistic)
            .reindex(index=CLASS_LABELS, columns=combination_order)
        )
        if pivot.isna().any().any():
            raise ValueError(f"Missing per-class recall cells for probe={probe}, statistic={statistic}")

        fig, ax = plt.subplots(
            figsize=(max(8, 1.35 * len(combination_order)), max(5, 0.42 * NUM_CLASSES + 2.5))
        )
        image = ax.imshow(pivot.to_numpy(dtype=float), vmin=vmin, vmax=vmax, aspect="auto")
        ax.set_xticks(np.arange(len(combination_order)))
        ax.set_xticklabels(combination_order, rotation=30, ha="right")
        ax.set_yticks(np.arange(NUM_CLASSES))
        ax.set_yticklabels(CLASS_LABELS)
        ax.set_xlabel("Encoder combination")
        ax.set_ylabel("Class")
        ax.set_title(
            f"{probe}: per-class recall "
            + ("mean across training seeds" if statistic == "mean_recall" else "SD across training seeds")
        )
        fig.colorbar(image, ax=ax, label=statistic)

        if NUM_CLASSES <= ANNOTATE_CONFUSION_IF_CLASSES_LEQ and len(combination_order) <= 8:
            values = pivot.to_numpy(dtype=float)
            for i in range(values.shape[0]):
                for j in range(values.shape[1]):
                    ax.text(j, i, f"{values[i, j]:.2f}", ha="center", va="center", fontsize=7)

        fig.tight_layout()
        path = output_dir / f"per_class_recall_{probe}_{statistic}.png"
        fig.savefig(path, dpi=180, bbox_inches="tight")
        plt.show()
        saved[probe] = path

    return saved


if not PER_CLASS_RECALL.empty:
    PER_CLASS_RECALL_AGG = aggregate_per_class_recall(PER_CLASS_RECALL)
    PER_CLASS_RECALL_AGG.to_csv(OUTPUT_ROOT / "per_class_recall_aggregate.csv", index=False)
    display(PER_CLASS_RECALL_AGG.head(20))

    RECALL_MEAN_FIGURES = plot_per_class_recall_heatmaps(
        PER_CLASS_RECALL_AGG,
        statistic="mean_recall",
    )
    RECALL_SD_FIGURES = plot_per_class_recall_heatmaps(
        PER_CLASS_RECALL_AGG,
        statistic="sd_recall",
    )

## 10. Optional: compact report tables

This section creates presentation-friendly tables with `mean ± SD` for the selected metrics and per-class recall. These tables are convenient for copying into a report before deciding which figures to keep.

In [ ]:
def format_mean_sd(mean: float, sd: float, digits: int = 3) -> str:
    return f"{mean:.{digits}f} ± {sd:.{digits}f}"


if not METRIC_AGGREGATE.empty:
    formatted = METRIC_AGGREGATE.copy()
    formatted["mean ± SD"] = [
        format_mean_sd(float(mean), float(sd))
        for mean, sd in zip(formatted["mean"], formatted["sd"])
    ]
    METRIC_REPORT_TABLE = formatted.pivot_table(
        index=["metric", "combination"],
        columns="probe",
        values="mean ± SD",
        aggfunc="first",
    ).reindex(columns=list(PROBES))
    METRIC_REPORT_TABLE.to_csv(OUTPUT_ROOT / "selected_metric_report_table.csv")
    display(METRIC_REPORT_TABLE)

if 'PER_CLASS_RECALL_AGG' in globals() and not PER_CLASS_RECALL_AGG.empty:
    recall_formatted = PER_CLASS_RECALL_AGG.copy()
    recall_formatted["mean ± SD"] = [
        format_mean_sd(float(mean), float(sd))
        for mean, sd in zip(recall_formatted["mean_recall"], recall_formatted["sd_recall"])
    ]
    PER_CLASS_RECALL_REPORT = recall_formatted.loc[
        :, ["probe", "combination", "class_label", "mean ± SD"]
    ]
    PER_CLASS_RECALL_REPORT.to_csv(
        OUTPUT_ROOT / "per_class_recall_report_table.csv",
        index=False,
    )
    display(PER_CLASS_RECALL_REPORT.head(30))

## 11. Output checklist

After a complete benchmark, the top-level output directory should contain:

```text
benchmark_design.json
cross_combination_preflight.csv
master_results.csv
selected_metric_aggregate.csv
selected_metric_report_table.csv
per_class_recall_by_seed.csv
per_class_recall_aggregate.csv
per_class_recall_report_table.csv
comparison_figures/
    metric_*.png
    confusion_small_multiples_cnn_s_mean.png
    confusion_small_multiples_cnn_m_mean.png
    confusion_small_multiples_cnn_l_mean.png
    confusion_small_multiples_*_sd.png              # optional
    per_class_recall_cnn_s_mean_recall.png
    per_class_recall_cnn_m_mean_recall.png
    per_class_recall_cnn_l_mean_recall.png
    per_class_recall_*_sd_recall.png
<combination>/seed_<seed>/<probe>/
    summary.csv
    classification_splits.csv
    training_history.csv
    embeddings_test.npz
    ... standard Experiment C artifacts ...
```

### Recommended analysis language

For general metrics, report each probe separately as **mean ± SD across the five training seeds**. Do not interpret the SD as user-split uncertainty because the user split is intentionally fixed.

For confusion matrices:

- use the **mean row-normalized matrix** as the main small-multiples figure;
- use the SD matrix only as a stability diagnostic;
- use the **per-class recall heatmap** when the number of combinations becomes too large for easy visual comparison of full matrices.

If you later want inferential statistics, the paired seed design also makes it straightforward to compare combinations using seed-matched differences for a selected metric.